# exp6 — seed 재현 (GPU 4개, 한 라운드)

**논문의 헤드라인은 K=100 의 `BiMamba 65.0 vs ACM2 56.8 = +8.2` 인데 지금 seed 0 하나다.**
리뷰어가 제일 먼저 묻는 게 이거고, 8.2 가 seed 노이즈면 논문이 통째로 흔들린다.
diag 로 메커니즘을 아무리 규명해도 이게 안 받쳐 주면 소용없다.

GPU 4개면 **하룻밤(~8h)에 seed 1·2 를 둘 다** 돌릴 수 있다 — 2 태그 × 2 seed = 4 잡.
`cf.run_training_jobs` 가 잡마다 seed 를 받으므로 한 라운드에 들어간다.

| | |
|---|---|
| 학습 | `bimamba_pure`, `acm2` × seed 1, 2 → **4잡, ~8h** |
| eval | 같은 4셀 (K=100, s=10, n=500) → seed 당 ~2h |
| 결과 | seed 0/1/2 의 gap 평균 ± 표준편차 |

## 왜 이걸 먼저 하나

| 후보 | 코드 작업 | BiMamba 에 | 판정 |
|---|---|---|---|
| **seed 재현** | **없음** | 되면 **강해짐** | ⭐ 오늘 밤 |
| E2 `random` | 브랜치 머지 + 변형 등록 | 되면 강해짐 | 내일 |
| E1 임베딩 | 디코더 구현 | 되면 **약해질 수 있음** | 안 함 |

**seed 재현은 코드 작업이 0 이라 지금 바로 걸 수 있고, 유일하게 "잃을 게 없는" 실험이다.**
gap 이 재현되면 논문의 제일 약한 곳이 사라진다. 재현이 안 되면 — 그건 제출 전에
반드시 알아야 하는 것이다.

---

> ⚠️ 이 노트북은 `exp5_tonight.py` 를 부르기만 한다. 잡 정의·스킵·집계는 전부 거기 있다.
> 학습 셀은 ~8h 블로킹된다. 중간에 멈춰도 완료분은 skip 되므로 다시 실행하면 이어서 간다.

## 0) 부팅

In [ ]:
import sys
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()          # common_final reload + 태그 등록 (순서 중요)

print('TASK      :', X.TASK)
print('기본 SEED :', X.SEED)
print('MAIN_STRIDE:', X.MAIN_STRIDE, ' N_EP:', X.N_EP, '(task 당 -> overall 500)')
print('GPU       :', v23.available_gpus())

## 1) 설정

In [ ]:
GPUS  = [0, 1, 2, 3]          # GPU 4개
K     = 100                   # 헤드라인 셀
SEEDS = [1, 2]                # 새로 돌릴 seed (0 은 이미 있음)
VARIANTS = ['bimamba', 'acm2']  # exp5 의 변형 이름. bimamba -> tag 'bimamba_pure'

TAGS = [X.tag_of(v, K) for v in VARIANTS]
JOBS = [(t, s, X.TASK) for s in SEEDS for t in TAGS]

print('태그 :', TAGS)
print('잡   :', len(JOBS), '개 ->', JOBS)
print(f'예상 : GPU {len(GPUS)}개로 ~8h (잡당 ~8h, {len(JOBS)}잡 / {len(GPUS)}GPU = 1라운드)')

## 2) ⚠️ 먼저 dry-run 으로 커맨드 확인

**`bimamba_pure` 커맨드에 `--use_chunk_pairs` 가 없고 `--policy.sscp_enabled=false` 가
있어야 한다.** 순수 BiMamba 는 carry off 로 학습해야 하는데, 이게 틀리면 8시간을 날린다.

그리고 `--seed` 와 출력 경로의 `seed{N}` 이 의도한 값인지 같이 본다.

In [ ]:
for t, s, task in JOBS:
    cmd = v23.make_train_cmd(t, s, task, gpu_id=GPUS[0])
    print(f"--- {t}  seed{s} ---")
    print(' ', cmd)
    flags = {'chunk_pairs': '--use_chunk_pairs' in cmd,
             'sscp_false': '--policy.sscp_enabled=false' in cmd,
             f'seed{s}':    f'seed{s}' in cmd or f'--seed={s}' in cmd}
    print('  확인:', flags)
    if t.startswith('bimamba_pure') and (flags['chunk_pairs'] or not flags['sscp_false']):
        print('  !! 순수 BiMamba 인데 플래그가 틀렸다. 실행하지 말 것.')
    print()

## 3) 학습 — 4잡을 4 GPU 에 (~8h 블로킹)

이미 150k 인 잡은 skip, 중단된 것(PART)은 resume 한다. 걸어놓고 자면 된다.

In [ ]:
# ~8h 블로킹. 중간에 멈춰도 다시 실행하면 이어서 간다.
cf.run_training_jobs(JOBS, GPUS, prefetch_task=X.TASK)

## 4) eval — seed 별로 (seed 당 ~2h)

`run_evals` 는 모듈 레벨 `SEED` 를 쓰므로 seed 마다 갈아 끼우며 돈다.
완료분은 자동 skip. 끝나면 `SEED` 를 0 으로 되돌린다.

In [ ]:
EVAL_JOBS = [X._rm_job(v, K, X.MAIN_STRIDE) for v in VARIANTS]
print('eval 셀:', [j['out'] for j in EVAL_JOBS])

_orig = X.SEED
try:
    for s in SEEDS:
        print('\n' + '#' * 60 + f'\n# seed {s}\n' + '#' * 60)
        X.SEED = s
        X.run_evals(EVAL_JOBS, GPUS)
finally:
    X.SEED = _orig
    print('\nSEED 복구 ->', X.SEED)

## 5) 집계 — gap 이 재현되는가

이 표가 오늘 밤의 답이다. **gap 의 표준편차가 작고 평균이 8 근처면 헤드라인이 굳는다.**

In [ ]:
import statistics as st

ALL_SEEDS = [0] + SEEDS
rows = []
_orig = X.SEED
try:
    for s in ALL_SEEDS:
        X.SEED = s
        bi = X._sr(f'rm_bimamba_k{K}_s{X.MAIN_STRIDE}')
        ac = X._sr(f'rm_acm2_k{K}_s{X.MAIN_STRIDE}')
        rows.append((s, bi, ac, (bi - ac) if (bi is not None and ac is not None) else None))
finally:
    X.SEED = _orig

print(f"{'seed':>5} {'BiMamba':>9} {'ACM2':>8} {'gap':>8}")
print('-' * 33)
for s, bi, ac, g in rows:
    f = lambda x: f'{x:.1f}' if x is not None else '  --'
    print(f"{s:>5} {f(bi):>9} {f(ac):>8} {f(g):>8}")

gaps = [g for _, _, _, g in rows if g is not None]
if len(gaps) >= 2:
    m, sd = st.mean(gaps), (st.stdev(gaps) if len(gaps) > 1 else 0.0)
    print('-' * 33)
    print(f"{'평균':>5} {'':>9} {'':>8} {m:>8.1f}")
    print(f"{'표준편차':>5} {'':>9} {'':>8} {sd:>8.1f}")
    print()
    if sd < 0.3 * abs(m):
        print(f'>> gap {m:.1f} +- {sd:.1f} — 재현된다. 헤드라인 유지.')
    else:
        print(f'>> gap {m:.1f} +- {sd:.1f} — 흔들린다. seed 를 더 늘리거나 주장을 약하게 할 것.')
elif gaps:
    print('\n아직 seed 하나뿐이다. eval 이 덜 끝났는지 확인할 것.')

## 6) 다음 라운드에 뭘 넣나

### 우선순위

1. **`acm2_k150` · `acm2_k50` 를 150k 로** — diag2 에서 이 둘의 150k 체크포인트가
   **없다**고 나왔는데 논문 표에는 값이 있다 (55.6 / 45.4). 즉 표의 셀들이 서로 다른
   step 이다. 이걸 메우면 `AUTHOR CHECK: Audit run-specific step counts` 가 해결된다.
2. **E2 `bimamba_scan="random"`** — 코드에 이미 있으나 한 번도 안 돌았다.
   `random ~ 56` 이면 *"역방향(쿼리를 맨 앞으로)이 특별하다"* 가 되어 BiMamba 라는
   이름이 정당해진다. **선행조건: `ecd-bimamba` 브랜치 머지 + `VARIANTS` 에 항목 추가.**
3. **K=150 seed 1·2** — 두 번째 gap(+9.0)도 재현되면 더 세진다.

### 안 하는 것

**E1 (역방향을 `nn.Embedding(K,D)` 로 바꿔 scratch 학습).** 논문이 그 주장을 하지 않고
(`results.tex` 초안이 명시적으로 열린 질문으로 남긴다), 결과가 나쁘면 제출 직전에
프레이밍을 다시 잡아야 한다. 비대칭 리스크다.

### 다음 라운드 잡 만들기

`JOBS` 만 바꾸고 셀 2→3 을 다시 돌리면 된다:

```python
# 1번 — 빠진 step 메우기 (seed 0)
JOBS = [('acm2_k150', 0, X.TASK), ('acm2_k50', 0, X.TASK),
        ('bimamba_pure_k150', 1, X.TASK), ('acm2_k150', 1, X.TASK)]

# 3번 — K=150 seed 재현
JOBS = [(t, s, X.TASK) for s in (1, 2) for t in ('bimamba_pure_k150', 'acm2_k150')]
```

---

## 7) 읽는 법

| gap 결과 | 뜻 |
|---|---|
| 평균 ~8, 표준편차 < 2.5 | **재현. 헤드라인 유지.** 논문 제일 약한 곳이 사라진다 |
| 표준편차가 평균의 30% 이상 | 흔들린다. seed 를 늘리거나 주장을 약하게 |
| seed 1·2 에서 gap 이 반토막 | **제출 전에 알아서 다행인 경우.** 주장을 다시 잡아야 한다 |

어느 쪽이 나오든 **알고 내는 것과 모르고 내는 것은 다르다.** 이게 코드 작업 없이
하룻밤에 되는 유일한 실험이라 제일 먼저 거는 게 맞다.